In [ ]:
# %%
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os
import kagglehub
import joblib

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn import metrics

In [ ]:
# %%
path = kagglehub.dataset_download("sheemazain/house-price-predication")

files = os.listdir(path)
file_path = os.path.join(path, files[0])

df = pd.read_csv(file_path)

print(df.head())
print(df.info())

In [ ]:
# %%
if "date" in df.columns:
    df["date"] = pd.to_datetime(df["date"])
    df["year"] = df["date"].dt.year
    df.drop("date", axis=1, inplace=True)

if "street" in df.columns:
    df.drop("street", axis=1, inplace=True)

# Remove invalid values
df = df[df["price"] > 0]

# Remove outliers
df = df[df["price"] < df["price"].quantile(0.97)]

# Drop missing
df = df.dropna()

In [ ]:
# %%
if "yr_renovated" in df.columns:
    df["renovated"] = (df["yr_renovated"] > 0).astype(int)

df["total_sqft"] = df["sqft_living"] + df["sqft_basement"]
df["bed_bath_ratio"] = df["bedrooms"] / (df["bathrooms"] + 1)

In [ ]:
# %%
if "city" in df.columns:
    top_cities = df["city"].value_counts().nlargest(10).index
    df["city"] = df["city"].apply(lambda x: x if x in top_cities else "other")

df = pd.get_dummies(df, drop_first=True)

In [ ]:
# %%
corr_matrix = df.corr()
price_corr = corr_matrix["price"].sort_values(ascending=False)

top_features = price_corr.index[:10]

plt.figure(figsize=(8,6))
sns.heatmap(df[top_features].corr(), annot=True, cmap="plasma")
plt.title("Top Feature Correlation Heatmap")
plt.show()

important_features = price_corr[abs(price_corr) > 0.03].index

plt.figure(figsize=(8,6))
sns.heatmap(df[important_features].corr(), cmap="coolwarm")
plt.title("Strong Feature Correlation Heatmap")
plt.show()

In [ ]:
# %%
X = df.drop("price", axis=1)
X = X[important_features.drop("price")]

y_log = np.log(df["price"])
y = df["price"]

In [ ]:
corr_matrix = df.corr()
price_corr = corr_matrix["price"].sort_values(ascending=False)

print("\nCorrelation with Price:\n", price_corr)

important_features = price_corr[abs(price_corr) > 0.03].index

In [ ]:
# %%
X_train, X_test, y_train_log, y_test_log, y_train, y_test = train_test_split(
    X, y_log, y, test_size=0.2, random_state=42
)

# Align columns
X_test = X_test.reindex(columns=X_train.columns, fill_value=0)

print("Train shape:", X_train.shape)
print("Test shape:", X_test.shape)

In [ ]:
# %%
feature_names = X_train.columns

In [ ]:
# %%
scaler = StandardScaler()

X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

In [ ]:
# %%
model = LinearRegression()
model.fit(X_train, y_train_log)

In [ ]:
test_data_prediction = model.predict(X_test)
print(test_data_prediction)

In [ ]:
# %%
coeff_df = pd.DataFrame(model.coef_, feature_names, columns=["Coefficient"])
coeff_df = coeff_df.sort_values(by="Coefficient", ascending=False)

print("\nFeature Coefficients:\n")
print(coeff_df.head(10))

In [ ]:
# %%
y_pred_log = model.predict(X_test)

# Prevent overflow
y_pred_log = np.clip(y_pred_log, -10, 20)

y_pred = np.exp(y_pred_log)

In [ ]:
# %%
rmse = np.sqrt(metrics.mean_squared_error(y_test, y_pred))
r2 = metrics.r2_score(y_test, y_pred)

print("\nModel Performance:")
print("Train R2 (log):", model.score(X_train, y_train_log))
print("Test R2:", r2)
print("RMSE:", rmse)

In [ ]:
# %%
sample = X_test[0:1]
pred = np.exp(model.predict(sample))

print("Predicted Price:", pred[0])
print("Actual Price:", y_test.iloc[0])

In [ ]:
# %%
joblib.dump(model, "house_price_model.pkl")
print("Model saved successfully!")

In [ ]:
# %%
plt.figure(figsize=(6,6))
sns.regplot(x=y_test, y=y_pred, scatter_kws={"alpha":0.5})
plt.title("Actual vs Predicted Prices")
plt.show()

plt.figure(figsize=(6,6))
plt.scatter(y_test, y_pred, alpha=0.5)
plt.plot([y_test.min(), y_test.max()],
         [y_test.min(), y_test.max()],
         color='red')
plt.title("Perfect Prediction Line")
plt.show()

errors = y_test - y_pred

plt.figure(figsize=(6,4))
sns.histplot(errors, kde=True)
plt.title("Error Distribution")
plt.show()